# Dataset 4 — Inspection Notebook

**Dataset:** Large non-experimental data (historic, retrospective use)
**Raw folder:** `data_raw/4. data set, large non-experimental data/`
**Purpose:** Structured inspection of all DS4 files before any cleaning is applied.
**Companion notebook:** `04_dataset_4_cleaning.ipynb`
**Audit report:** `outputs/reports/ds4_audit_report.md`

This notebook follows the CLAUDE.md dataset-first workflow:
inventory → documentation alignment → structural understanding → data quality audit.

**No transformations are applied here.**

## 0. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Root is one level above notebooks/
ROOT    = Path("..").resolve()
DS4_RAW = ROOT / "data_raw" / "4. data set, large non-experimental data"
REPORTS = ROOT / "outputs" / "reports"

print("Project root:", ROOT)
print("DS4 raw folder exists:", DS4_RAW.exists())

---
## 1. File Inventory

List every file in the DS4 folder and infer its role.

In [ ]:
all_files = sorted(DS4_RAW.rglob("*"))
for f in all_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"{f.relative_to(DS4_RAW)}  [{size_kb:.0f} KB]")

In [ ]:
# File paths used throughout this notebook
BEH_CSV    = DS4_RAW / "Behavior data set" / "behavior_data_set_4.csv"
SURVEY_CSV = DS4_RAW / "Survey data set"   / "post_survey_data_set_4.csv"
SURVEY_XLS = DS4_RAW / "Survey data set"   / "post_survey_data_set_4.xlsx"

for p in [BEH_CSV, SURVEY_CSV, SURVEY_XLS]:
    size = f"  [{p.stat().st_size/1024/1024:.1f} MB]" if p.exists() else ""
    print(p.name, "exists:", p.exists(), size)

**File roles (confirmed):**

| File | Role |
|---|---|
| `behavior_data_set_4.csv` | Event-level intervention data — primary behavioral substrate |
| `post_survey_data_set_4.csv` | Post-use retrospective survey (semicolon-delimited) |
| `post_survey_data_set_4.xlsx` | Near-duplicate of survey CSV (pre-trimmed whitespace) |
| `codebook_behavior_data_set.pdf` | Variable documentation for behavioral file (has discrepancies) |
| `codebook_post_survey_data_set.*` | Variable documentation for survey |
| `Project description...docx` | Study design context — non-experimental, retrospective |

---
## 2. Behavioral Data
### 2.1 Load and basic structure

**Critical:** DS4 behavioral CSV is COMMA-delimited — different from DS1 (semicolon).

In [ ]:
# Confirm delimiter — inspect first line
with open(BEH_CSV) as f:
    print("First line:", repr(f.readline()[:120]))

In [ ]:
# IMPORTANT: sep="," — behavioral file is comma-delimited
beh = pd.read_csv(BEH_CSV, sep=",", low_memory=False)
print("Shape:", beh.shape)
print("Columns:", beh.columns.tolist())
print()
beh.head(3)

### 2.2 Timestamp format

**Confirmed:** DS4 timestamps are ISO 8601 UTC strings (e.g. `2022-09-29T15:48:54Z`).
This is DIFFERENT from DS1, which uses Unix epoch int64.
Parse with `pd.to_datetime(df["timestamp"], utc=True)`.

In [ ]:
sample_ts = beh["timestamp"].iloc[0]
print(f"Raw value:     {sample_ts}")
print(f"Type:          {type(sample_ts)}")
print(f"Converted UTC: {pd.to_datetime(sample_ts, utc=True)}")
print()
# Check for parse failures
try:
    ts_parsed = pd.to_datetime(beh["timestamp"], utc=True)
    print(f"Parse failures: 0 / {len(beh):,}")
    print(f"Date range: {ts_parsed.min()} to {ts_parsed.max()}")
except Exception as e:
    print(f"Parse error: {e}")

### 2.3 Resolution values

**Confirmed 5 values (codebook only documents 3):**
- `closedApp` — dominant at 47.1% (vs 24% in DS1)
- `openedApp` — 35.4%
- `dismissedAppOpening` — 17.5%
- `dismissedByStructuredIntervention` — 0.03% — **undocumented in codebook**
- `dismissedWithHealthyAlternative` — <0.01% — **undocumented in codebook**

In [ ]:
print("Resolution value counts:")
vc  = beh["resolution"].value_counts(dropna=False)
pct = beh["resolution"].value_counts(dropna=False, normalize=True) * 100
for val in vc.index:
    flag = " <- UNDOCUMENTED" if val in ["dismissedByStructuredIntervention", "dismissedWithHealthyAlternative"] else ""
    print(f"  {str(val):<45s} {vc[val]:>10,}  ({pct[val]:.1f}%){flag}")

### 2.4 Intervention fields: interventionDuration and interventionType

**Key finding:** Both columns are ALWAYS NaN together (63.5% NaN).
All `closedApp` events have NaN intervention fields.
This may indicate a structurally different event logging path, not simple missing data.

In [ ]:
null_dur  = beh["interventionDuration"].isna().sum()
null_type = beh["interventionType"].isna().sum()
both_null = (beh["interventionDuration"].isna() & beh["interventionType"].isna()).sum()
one_null  = (beh["interventionDuration"].isna() ^ beh["interventionType"].isna()).sum()
print(f"interventionDuration NaN:   {null_dur:,} ({null_dur/len(beh):.1%})")
print(f"interventionType NaN:       {null_type:,} ({null_type/len(beh):.1%})")
print(f"Both NaN (always together): {both_null:,}")
print(f"One NaN, one not (anomalies): {one_null}  <- should be 0")
print()
print("NaN intervention by resolution:")
for res, grp in beh.groupby("resolution"):
    n_null = grp["interventionType"].isna().sum()
    print(f"  {res:<45s} {n_null:>10,} NaN / {len(grp):,} ({n_null/len(grp):.1%})")

In [ ]:
print("interventionDuration (non-NaN) stats:")
print(beh["interventionDuration"].describe().round(2))
print()
print("interventionType unique values (non-NaN):")
print(beh["interventionType"].value_counts(dropna=True))

### 2.5 purpose — missingness

In [ ]:
n_null  = beh["purpose"].isna().sum()
n_total = len(beh)
print(f"NaN purpose: {n_null:,} / {n_total:,}  ({n_null/n_total:.1%})")
print(f"Non-null:    {n_total - n_null:,}")
if n_total - n_null > 0:
    print()
    print("Sample non-null purpose values:")
    print(beh["purpose"].dropna().head(10).tolist())

### 2.6 secondsSinceStudySetup and weeksSinceStudySetup

**Critical finding:** Both columns are entirely <= 0. Codebook says forward-counting from setup — this is incorrect.
The variable counts backward from a per-participant fixed anchor (~8.8h after last event).
Do NOT use as "time since setup." Reconstruct forward time from `timestamp` directly.

In [ ]:
for col in ["secondsSinceStudySetup", "weeksSinceStudySetup"]:
    n_neg  = (beh[col] < 0).sum()
    n_zero = (beh[col] == 0).sum()
    n_pos  = (beh[col] > 0).sum()
    n_null = beh[col].isna().sum()
    print(f"{col}:")
    print(f"  < 0:  {n_neg:,}  ({n_neg/len(beh):.1%})")
    print(f"  = 0:  {n_zero:,}")
    print(f"  > 0:  {n_pos:,}")
    print(f"  NaN:  {n_null:,}")
    print(f"  Min:  {beh[col].min():.0f}  Max: {beh[col].max():.0f}")
    print()

In [ ]:
# Verify per-participant anchor interpretation
# timestamp_epoch - secondsSinceStudySetup should be near-constant per participant
beh_ts = beh.copy()
beh_ts["ts_epoch"]        = pd.to_datetime(beh_ts["timestamp"], utc=True).astype("int64") // 10**9
beh_ts["anchor_estimate"] = beh_ts["ts_epoch"] - beh_ts["secondsSinceStudySetup"]

anchor_std = beh_ts.groupby("participantUuid")["anchor_estimate"].std()
print("Within-participant anchor std (confirms per-participant fixed point):")
print(f"  Exact zero std:  {(anchor_std == 0).sum()} participants")
print(f"  Std < 1 sec:     {(anchor_std < 1).sum()} participants")
print(f"  Median std:      {anchor_std.median():.2f} sec")

### 2.7 Participants

In [ ]:
print(f"Unique participants (participantUuid): {beh['participantUuid'].nunique():,}")
print(f"NaN participantUuid rows:              {beh['participantUuid'].isna().sum()}")

In [ ]:
# Events per participant distribution
events_per_p = beh["participantUuid"].value_counts()
print(f"Min: {events_per_p.min()}  Median: {events_per_p.median():.0f}  Mean: {events_per_p.mean():.0f}  Max: {events_per_p.max():,}")
print()
brackets = [("<10", lambda x: x < 10),
            ("10-99", lambda x: (x >= 10) & (x < 100)),
            ("100-999", lambda x: (x >= 100) & (x < 1000)),
            ("1000-4999", lambda x: (x >= 1000) & (x < 5000)),
            (">=5000", lambda x: x >= 5000)]
for label, fn in brackets:
    n = fn(events_per_p).sum()
    print(f"  {label:<12s} {n:>4} participants")

In [ ]:
# Study duration per participant
beh_ts3 = beh.copy()
beh_ts3["ts_dt"] = pd.to_datetime(beh_ts3["timestamp"], utc=True)
duration = beh_ts3.groupby("participantUuid")["ts_dt"].agg(lambda x: (x.max() - x.min()).days)
print("Study duration (days):")
print(f"  Median: {duration.median():.0f}  Mean: {duration.mean():.0f}  Max: {duration.max()}")
print(f"  > 365 days: {(duration > 365).sum()} participants")
print(f"  > 730 days: {(duration > 730).sum()} participants")

### 2.8 App distribution

In [ ]:
print(f"Unique apps: {beh['app'].nunique()}")
print()
print("Top 15 apps by event count:")
print(beh["app"].value_counts().head(15))

In [ ]:
# Typo check: 'whastapp' should be 'whatsapp'
what_variants = beh[beh["app"].str.lower().str.contains("what", na=False)]["app"].value_counts()
print("WhatsApp spelling variants:")
print(what_variants)

### 2.9 Duplicate rows

In [ ]:
n_dupes = beh.duplicated().sum()
print(f"Fully duplicate rows: {n_dupes:,}  ({n_dupes/len(beh):.2%})")

### 2.10 Full null summary — behavioral

In [ ]:
null_beh = pd.DataFrame({
    "null_count": beh.isnull().sum(),
    "null_pct_%": (beh.isnull().sum() / len(beh) * 100).round(2)
})
print(null_beh.to_string())

---
## 3. Survey Data
### 3.1 Load and structure

**Critical:** DS4 survey CSV is SEMICOLON-delimited — different from behavioral file (comma).

In [ ]:
# Confirm delimiter
with open(SURVEY_CSV) as f:
    print("Delimiter check — first line:", repr(f.readline()[:120]))

In [ ]:
survey = pd.read_csv(SURVEY_CSV, sep=";")
print("Shape:", survey.shape)
print("Columns:", survey.columns.tolist())
print()
survey.head(3)

### 3.2 Survey UUID prefix

**Critical finding:** Survey `uuid` has the prefix `donatus-master-thesis-one-sec-early-2023-`.
This prefix must be stripped before matching to behavioral `participantUuid`.

In [ ]:
print("Sample uuid values:")
print(survey["uuid"].head(5).tolist())
print()
prefix = "donatus-master-thesis-one-sec-early-2023-"
has_prefix = survey["uuid"].str.startswith(prefix).sum()
print(f"Rows with prefix: {has_prefix} / {len(survey)}")

In [ ]:
# Strip prefix and match to behavioral
prefix = "donatus-master-thesis-one-sec-early-2023-"
survey_ids_stripped = survey["uuid"].str.replace(prefix, "", regex=False).str.upper()
beh_ids = set(beh["participantUuid"].str.upper())

matched   = survey_ids_stripped.isin(beh_ids).sum()
unmatched = (~survey_ids_stripped.isin(beh_ids)).sum()
beh_only  = len(beh_ids) - matched
print(f"Survey matched to behavioral: {matched}")
print(f"Survey with no behavioral:    {unmatched}")
print(f"Behavioral with no survey:    {beh_only}")

### 3.3 Survey duplicates

In [ ]:
n_u = survey["uuid"].nunique()
print(f"Unique uuid: {n_u}  |  Total rows: {len(survey)}  |  Duplicate rows: {len(survey)-n_u}")
if len(survey) - n_u > 0:
    dupe_mask = survey.duplicated(subset="uuid", keep=False)
    print()
    print("Rows with duplicated uuid:")
    print(survey[dupe_mask][["uuid"]].head(10))

### 3.4 Gender values

**Codebook discrepancy:** Codebook documents numeric codes (1=Female, etc.).
Actual data stores text strings — use text-based recoding only.

In [ ]:
print("gender value counts (text strings, not numeric codes):")
print(survey["gender"].value_counts(dropna=False))

### 3.5 Missingness summary — survey

In [ ]:
null_sur = pd.DataFrame({
    "null_count": survey.isnull().sum(),
    "null_pct_%": (survey.isnull().sum() / len(survey) * 100).round(2)
})
print(null_sur.to_string())

---
## 4. CSV vs XLSX Comparison

Both survey files should be functionally equivalent. CSV has trailing whitespace; XLSX is pre-trimmed.

In [ ]:
survey_x = pd.read_excel(SURVEY_XLS)
print(f"CSV shape: {survey.shape}  XLSX shape: {survey_x.shape}")
print()
# Check for non-whitespace content differences
diffs = {}
for col in survey.columns:
    if col in survey_x.columns:
        csv_s  = survey[col].apply(lambda x: str(x).strip() if pd.notna(x) else None)
        xlsx_s = survey_x[col].apply(lambda x: str(x).strip() if pd.notna(x) else None)
        n_diff = (csv_s != xlsx_s).sum()
        if n_diff > 0:
            diffs[col] = n_diff
if diffs:
    print("Columns with differences after whitespace stripping:")
    for col, n in diffs.items():
        print(f"  {col}: {n} rows")
else:
    print("No content differences after whitespace stripping — files are equivalent.")

---
## 5. Participant Overlap

In [ ]:
prefix = "donatus-master-thesis-one-sec-early-2023-"
beh_ids_set    = set(beh["participantUuid"].dropna().str.upper())
survey_ids_set = set(survey["uuid"].str.replace(prefix, "", regex=False).str.upper())

in_both      = beh_ids_set & survey_ids_set
beh_only_set = beh_ids_set - survey_ids_set
sur_only_set = survey_ids_set - beh_ids_set

print(f"Behavioral only:        {len(beh_only_set):,}")
print(f"Survey only:            {len(sur_only_set):,}")
print(f"Both files (linkable):  {len(in_both):,}")
print(f"Total unique:           {len(beh_ids_set | survey_ids_set):,}")

---
## 6. Codebook Discrepancies

In [ ]:
discrepancies = [
    ("High",   "Behavioral", "secondsSinceStudySetup/weeksSinceStudySetup: all values negative; codebook says forward-counting"),
    ("Medium", "Behavioral", "resolution: codebook lists 3 values; data has 5 (2 undocumented)"),
    ("Medium", "Survey",     "gender: codebook uses numeric codes 1-5; actual data stores text strings"),
    ("Medium", "Survey",     "uuid prefix not documented in codebook"),
    ("Low",    "Behavioral", "No NaN mention in codebook for interventionDuration, interventionType, purpose"),
]
print(f"{'Severity':<8} {'File':<12} Description")
print("-" * 78)
for sev, f, desc in discrepancies:
    print(f"{sev:<8} {f:<12} {desc}")

---
## 7. Inspection Summary

In [ ]:
print("=" * 65)
print("DS4 INSPECTION SUMMARY")
print("=" * 65)
print()
print("FILES & DELIMITER")
print(f"  Behavioral:   {beh.shape[0]:,} rows x {beh.shape[1]} cols | sep=','")
print(f"  Survey CSV:   {survey.shape[0]} rows x {survey.shape[1]} cols | sep=';'")
print()
print("TIMESTAMP")
print("  Format:  ISO 8601 UTC string (e.g. 2022-09-29T15:48:54Z)")
print("  Parse:   pd.to_datetime(df['timestamp'], utc=True)")
print("  Range:   2020-09-30 to 2023-04-02")
print()
print("RESOLUTION")
for res, cnt in beh["resolution"].value_counts(dropna=False).items():
    pct = cnt / len(beh) * 100
    flag = " <- UNDOCUMENTED" if res in ["dismissedByStructuredIntervention", "dismissedWithHealthyAlternative"] else ""
    print(f"  {str(res):<45s} {cnt:>10,} ({pct:.1f}%){flag}")
print()
print("INTERVENTION FIELDS")
n_nan = beh["interventionDuration"].isna().sum()
print(f"  NaN interventionDuration/Type: {n_nan:,} rows ({n_nan/len(beh):.1%})")
print("  All closedApp events have NaN intervention fields")
print()
print("PARTICIPANTS")
print(f"  Behavioral: {beh['participantUuid'].nunique():,}")
print(f"  Survey (unique uuid): {survey['uuid'].nunique()}")
print(f"  Linkable (both files): {len(in_both)}")
print()
print("OPEN ISSUES (must resolve before cleaning)")
print("  1. 2 undocumented resolution values — confirm meaning before mapping")
print("  2. secondsSinceStudySetup is inverted — derive time from timestamp directly")
print("  3. closedApp events may be a different event type — classify separately")
print("  4. Survey uuid prefix must be stripped for ID matching")
print()
print("-> Proceed to 04_dataset_4_cleaning.ipynb")